<a href="https://colab.research.google.com/github/srihanreddy/Reinforcement-Learning/blob/main/Lab_Assignment_6_Gymnasium_Environments%E2%80%94CliffWalking%E2%80%94Taxi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Installation
First, we need to install the standard `gymnasium` library to access the environments.

In [1]:
!pip install gymnasium

### 2. Implementation of the Random Agent
We will define a class `RandomAgent` that takes an action space and samples randomly from it. We'll also define a helper function to run episodes and calculate rewards.

In [2]:
import gymnasium as gym
import numpy as np

class RandomAgent:
    def __init__(self, action_space):
        self.action_space = action_space

    def select_action(self, state):
        # Select a random action from the environment's action space
        return self.action_space.sample()

def run_random_agent_experiment(env_name, num_episodes=5):
    print(f"\n--- Starting Experiment on {env_name} ---")
    env = gym.make(env_name)
    agent = RandomAgent(env.action_space)

    all_episode_rewards = []

    for episode in range(1, num_episodes + 1):
        state, info = env.reset()
        done = False
        truncated = False
        total_reward = 0
        steps = 0

        while not (done or truncated):
            action = agent.select_action(state)
            next_state, reward, done, truncated, info = env.step(action)

            total_reward += reward
            steps += 1
            state = next_state

        all_episode_rewards.append(total_reward)
        avg_reward = total_reward / steps if steps > 0 else 0
        print(f"Episode {episode}: Total Reward = {total_reward}, Steps = {steps}, Avg Reward/Step = {avg_reward:.2f}")

    env.close()

    overall_avg_reward = np.mean(all_episode_rewards)
    print(f"\nSummary for {env_name} over {num_episodes} episodes:")
    print(f"Average Total Reward per Episode: {overall_avg_reward:.2f}")

### 3. Run Experiments
Now, let's run our Random Agent on **CliffWalking-v0** and **Taxi-v3**.

In [5]:
# Run experiment on CliffWalking (updated to v1 to resolve deprecation error)
run_random_agent_experiment("CliffWalking-v1", num_episodes=5)

# Run experiment on Taxi (updated to v4 to resolve deprecation error)
run_random_agent_experiment("Taxi-v4", num_episodes=5)


--- Starting Experiment on CliffWalking-v1 ---
Episode 1: Total Reward = -106263, Steps = 10233, Avg Reward/Step = -10.38
Episode 2: Total Reward = -370621, Steps = 38278, Avg Reward/Step = -9.68
Episode 3: Total Reward = -979, Steps = 88, Avg Reward/Step = -11.12
Episode 4: Total Reward = -12147, Steps = 960, Avg Reward/Step = -12.65
Episode 5: Total Reward = -24932, Steps = 2954, Avg Reward/Step = -8.44

Summary for CliffWalking-v1 over 5 episodes:
Average Total Reward per Episode: -102988.40

--- Starting Experiment on Taxi-v4 ---
Episode 1: Total Reward = -758, Steps = 200, Avg Reward/Step = -3.79
Episode 2: Total Reward = -776, Steps = 200, Avg Reward/Step = -3.88
Episode 3: Total Reward = -785, Steps = 200, Avg Reward/Step = -3.92
Episode 4: Total Reward = -785, Steps = 200, Avg Reward/Step = -3.92
Episode 5: Total Reward = -839, Steps = 200, Avg Reward/Step = -4.20

Summary for Taxi-v4 over 5 episodes:
Average Total Reward per Episode: -788.60


### 4. Summary & Analysis of Results

#### Why does the Random Agent perform poorly on **CliffWalking-v1**?
* **The Catastrophic Cliff Penalty (-100):** The bottom row between the start and the goal contains the cliff. Whenever the agent randomly steps onto the cliff, it receives a heavy penalty of `-100` and gets sent all the way back to the starting cell.
* **No Step Limits / Long Walks:** Since there is no maximum step limit in CliffWalking, the agent can walk aimlessly for thousands of steps (e.g., over 30,000 steps), repeatedly falling off the cliff and accumulating enormous negative rewards (such as `-106,263` or `-370,621`).

#### Why does the Random Agent perform poorly on **Taxi-v4**?
* **Step Limit Truncation:** The Taxi environment has a strict step limit of `200` steps per episode. Because the random agent has no strategy, it never successfully picks up the passenger and drops them off at the destination. It simply runs out of time after exactly `200` steps.
* **Illegal Action Penalties:** The agent frequently receives a `-10` penalty for executing illegal `pickup` or `dropoff` actions when it is not at the correct grid coordinates.